# Analysis of our Transformer model

In [1]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import sys
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, roc_auc_score
from tqdm import tqdm
from pathlib import Path
from argparse import Namespace

module_path = Path("src").resolve()
sys.path.append(str(module_path))
from load_data import ProteinDataset, collate_fn, analyze_dataset
from new_file import custom_classifier
import classifier

## Set Parameters

In [2]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_LEN = 128
BATCH_SIZE = 32

# Classes in dataset
label_cols = ['NO_S', 'LIP', 'S', 'TATLIP', 'PILI', 'TA']

args = {
    "max_len": MAX_LEN,
    "vocab_size": 23,
    "num_classes": len(label_cols),
    "num_heads": 16,
    "num_layers": 8,
    "embed_dim": 256,
    "attention_fn": "dot_prod",
    "Classifier": classifier.LinearClassifier,
    "classifier_reduction": "mean"
}
args = Namespace(**args)

## Load Dataset

In [3]:
# Load test dataframe
df = pd.read_csv("data/dataset_clean.csv")
analyze_dataset(df, label_cols=label_cols, sequence_col='sequence')

===== DATASET SUMMARY =====
Total rows: 25693
Total columns: 10

===== SEQUENCE LENGTH STATISTICS =====
Min length:   29
Max length:   69
Mean length:  68.90
Median length:69.00

===== LABEL DISTRIBUTION =====
NO_S                     : 19036
S                        : 3652
LIP                      : 2261
TA                       : 595
PILI                     : 113
TATLIP                   : 36

===== DONE =====


## Create Test dataset

In [4]:
# Create test data set, with same random state as during training
_, test_df = train_test_split(df, test_size=0.1, random_state=42, shuffle=True)
analyze_dataset(test_df, label_cols=label_cols, sequence_col='sequence')

# Dataset and DataLoader
test_dataset = ProteinDataset(test_df, label_cols, seq_col="sequence", max_len=MAX_LEN)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

===== DATASET SUMMARY =====
Total rows: 2570
Total columns: 10

===== SEQUENCE LENGTH STATISTICS =====
Min length:   29
Max length:   69
Mean length:  68.91
Median length:69.00

===== LABEL DISTRIBUTION =====
NO_S                     : 1921
S                        : 359
LIP                      : 215
TA                       : 62
PILI                     : 7
TATLIP                   : 6

===== DONE =====


## Load Model

In [5]:
# Recreate model
model = custom_classifier(args)
model.load_state_dict(torch.load("model/signalP_dot_prod_mean_best_model.pt", map_location=DEVICE))
model.to(DEVICE)
model.eval()

custom_classifier(
  (transformer): SmallTransformer(
    (embed): Embedding(23, 256)
    (blocks): ModuleList(
      (0-7): 8 x TransformerEncoderBlock(
        (self_attn): MultiHeadedAttention(
          (Wq): Linear(in_features=256, out_features=256, bias=True)
          (Wk): Linear(in_features=256, out_features=256, bias=True)
          (Wv): Linear(in_features=256, out_features=256, bias=True)
          (Wo): Linear(in_features=256, out_features=256, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (ffn): Sequential(
          (0): Linear(in_features=256, out_features=1024, bias=True)
          (1): ReLU()
          (2): Linear(in_features=1024, out_features=256, bias=True)
        )
        (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (classifier): LinearClassifier(
    (classifier): Linear(i

## Run Model

In [6]:
all_preds = []
all_labels = []
all_emb = []

# Wrap the DataLoader with tqdm
test_loop = tqdm(test_loader, desc="Testing", leave=True)

# Run Model
with torch.no_grad():
    for batch in test_loop:
        sequences, labels, attention_mask = [b.to(DEVICE) for b in batch]
        logits, emb = model(sequences, attention_mask)
        all_preds.append(torch.sigmoid(logits).cpu())
        all_labels.append(labels.cpu())
        all_emb.append(emb.cpu())

        # Show running number of batches
        test_loop.set_postfix({"batch": f"{len(all_labels)}/{len(test_loader)}"})

all_preds = torch.cat(all_preds)
all_labels = torch.cat(all_labels)
all_emb = torch.cat(all_emb)

Testing: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 81/81 [01:46<00:00,  1.31s/it, batch=81/81]


### Save results

In [7]:
torch.save({
    "labels": all_labels,
    "predictions": all_preds,
    "embeddings": all_emb
}, "data/results/dot_prod_results.pt")

### Load saved results

In [8]:
data = torch.load("data/results/dot_prod_results.pt")
all_labels = data["labels"]
all_preds = data["predictions"]
all_emb = data["embeddings"]

### Evaluate results

In [9]:
# Binary predictions (threshold 0.5)
preds_bin = (all_preds >= 0.5).float()

# Macro F1
macro_f1 = f1_score(all_labels, preds_bin, average="macro", zero_division=0)
print("Test Macro F1:", macro_f1)

# Per-class F1 / ROC-AUC
for i, label in enumerate(label_cols):
    f1 = f1_score(all_labels[:, i], preds_bin[:, i], zero_division=0)
    try:
        auc = roc_auc_score(all_labels[:, i], all_preds[:, i])
    except ValueError:
        auc = float('nan')
    print(f"{label:20s} - F1: {f1:.4f} - ROC-AUC: {auc:.4f}")

Test Macro F1: 0.7860987961698487
NO_S                 - F1: 0.9757 - ROC-AUC: 0.9814
LIP                  - F1: 0.8838 - ROC-AUC: 0.9897
S                    - F1: 0.8455 - ROC-AUC: 0.9709
TATLIP               - F1: 0.4615 - ROC-AUC: 0.9502
PILI                 - F1: 0.7500 - ROC-AUC: 0.9692
TA                   - F1: 0.8000 - ROC-AUC: 0.9694


In [10]:
print("Model summary:")
print(f"Labels are of shape:\t\t{all_labels.shape}")
print(f"Predictions are of shape:\t{all_preds.shape}")
print(f"Embeddings are of shape:\t{all_emb.shape}")

Model summary:
Labels are of shape:		torch.Size([2570, 6])
Predictions are of shape:	torch.Size([2570, 6])
Embeddings are of shape:	torch.Size([2570, 256])


## Plot results